# Repository guide: Generic XLS-R; eight epochs; no augmentation

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# XLS-R 300M fine-tuning on Tarifit Corpus V1.2

This notebook runs a fresh `facebook/wav2vec2-xls-r-300m` CTC model on the frozen V1.2 train/validation partitions.

**Experiment policy**

- The V1.2 train/validation metadata are reused exactly as frozen for the other controlled experiments.
- The final test partition is not loaded for model selection or training.
- The tokenizer is rebuilt specifically for the V1.2 final Tarifit character inventory.
- The default condition is **no augmentation**, so it can be compared with the main MMS, OmniASR, and Fadhma V1.2 adaptation references.
- This diagnostic copy is fixed to the no-augmentation condition so it remains directly comparable with the no-augmentation MMS, OmniASR, and Fadhma runs.
- This diagnostic run is trained for all eight epochs without early stopping; the best checkpoint is selected afterward using validation CER.
        

In [ ]:
# Cell 1 — Install reproducible XLS-R V1.2 dependencies

!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 158.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Cell 2 — Mount Google Drive and define XLS-R V1.2 paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

# This notebook is the no-augmentation, full-eight-epoch diagnostic run.
# It uses the same V1.2 data and hyperparameters as the previous run.
AUGMENTATION_CONDITION = "noaug"

assert AUGMENTATION_CONDITION == "noaug"

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2.csv"
)

FROZEN_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2_train_val_frozen.csv"
)

TOKENIZER_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer_v1_2"
)

DATASET_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_corpus_v1_2"
)

EXPERIMENT_NAME = "xlsr_300m_tarifit_v1_2_noaug_full8"

MODEL_DIR = PROJECT_ROOT / "models" / EXPERIMENT_NAME
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME

BASE_MODEL_ID = "facebook/wav2vec2-xls-r-300m"
SEED = 42

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_ROOT.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Frozen train/validation metadata exists:", FROZEN_METADATA_PATH.exists())
print("Augmentation condition:", AUGMENTATION_CONDITION)
print("Experiment name:", EXPERIMENT_NAME)
print("Project root:", PROJECT_ROOT)
print("Model output:", MODEL_DIR)
print("Results output:", RESULTS_DIR)


Mounted at /content/drive
Project exists: True
Metadata exists: True
Frozen train/validation metadata exists: True
Augmentation condition: noaug
Experiment name: xlsr_300m_tarifit_v1_2_noaug_full8
Project root: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
Model output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug_full8
Results output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug_full8


In [ ]:
# Cell 3 — Record software and GPU environment

import sys
import random
import hashlib

import torch
import transformers
import datasets
import jiwer
import soundfile
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("JiWER:", jiwer.__version__ if hasattr(jiwer, "__version__") else "unknown")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    properties = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
JiWER: unknown
CUDA available: True
GPU: NVIDIA L4
GPU memory (GB): 22.03


In [ ]:
# Cell 4 — Load and verify the frozen V1.2 train/validation partitions

import pandas as pd

if not FROZEN_METADATA_PATH.exists():
    raise FileNotFoundError(
        "The frozen V1.2 train/validation metadata was not found. "
        "Do not train until the same frozen split used by the controlled experiments is available."
    )

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

required_columns = {
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "dataset_split",
    "duration_seconds",
    "audio_path",
    "transcription",
}

missing_columns = required_columns - set(frozen_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

frozen_df["dataset_split"] = (
    frozen_df["dataset_split"].astype(str).str.lower().str.strip()
)
frozen_df["transcription"] = (
    frozen_df["transcription"].fillna("").astype(str).str.strip()
)

assert frozen_df["dataset_split"].isin(["train", "validation"]).all()
assert frozen_df["transcription"].ne("").all()
assert not frozen_df["segment_id"].duplicated().any()

train_df = frozen_df[frozen_df["dataset_split"] == "train"].copy()
validation_df = frozen_df[
    frozen_df["dataset_split"] == "validation"
].copy()

print("Train segments:", len(train_df))
print("Validation segments:", len(validation_df))
print(
    "Train duration:",
    round(train_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print(
    "Validation duration:",
    round(validation_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print("Train speakers:", sorted(train_df["speaker_group_id"].unique()))
print(
    "Validation speakers:",
    sorted(validation_df["speaker_group_id"].unique()),
)

full_metadata_df = pd.read_csv(METADATA_PATH)
full_metadata_df["dataset_split"] = (
    full_metadata_df["dataset_split"].astype(str).str.lower().str.strip()
)

train_speakers = set(train_df["speaker_group_id"].dropna())
validation_speakers = set(validation_df["speaker_group_id"].dropna())
test_speakers = set(
    full_metadata_df.loc[
        full_metadata_df["dataset_split"] == "test",
        "speaker_group_id",
    ].dropna()
)

assert not train_speakers & validation_speakers
assert not train_speakers & test_speakers
assert not validation_speakers & test_speakers

missing_audio = [
    str(PROJECT_ROOT / path)
    for path in frozen_df["audio_path"]
    if not (PROJECT_ROOT / path).exists()
]

print("Missing audio files:", len(missing_audio))
assert not missing_audio, (
    "Some frozen audio files are missing. First missing file: "
    + (missing_audio[0] if missing_audio else "")
)

metadata_sha256 = hashlib.sha256(
    FROZEN_METADATA_PATH.read_bytes()
).hexdigest()

print("Frozen metadata SHA-256:", metadata_sha256)
print("Speaker-independent partition verification completed.")


Train segments: 1754
Validation segments: 129
Train duration: 5.223 hours
Validation duration: 0.298 hours
Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']
Missing audio files: 0
Frozen metadata SHA-256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab
Speaker-independent partition verification completed.


In [ ]:
# Cell 5 — Verify the V1.2 orthographic character inventory

from collections import Counter
import unicodedata

FINAL_LETTERS = (
    list("abcdefghijklmnpqrstuvwxyz")
    + ["ɛ", "ɣ", "ʷ", "ḍ", "ḥ", "ṭ"]
)

ALLOWED_CHARACTERS = set(FINAL_LETTERS) | {" "}

all_text = " ".join(frozen_df["transcription"].tolist())
character_counts = Counter(all_text)

unexpected_characters = {
    character: count
    for character, count in character_counts.items()
    if character not in ALLOWED_CHARACTERS
}

print("Expected letters:", " ".join(FINAL_LETTERS))
print("Number of letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected_characters)

for text in frozen_df["transcription"]:
    assert text == unicodedata.normalize("NFC", text)

assert not unexpected_characters, (
    "The V1.2 transcripts contain characters outside the declared vocabulary."
)

print("V1.2 character inventory verified.")


Expected letters: a b c d e f g h i j k l m n p q r s t u v w x y z ɛ ɣ ʷ ḍ ḥ ṭ
Number of letters: 31
Unexpected characters: {}
V1.2 character inventory verified.


In [ ]:
# Cell 6 — Build and save the XLS-R V1.2 CTC tokenizer

import json
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

vocab_tokens = ["|"] + FINAL_LETTERS + ["[UNK]", "[PAD]"]
vocab_dict = {
    token: index
    for index, token in enumerate(vocab_tokens)
}

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(vocab_dict, file, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained(TOKENIZER_DIR)

print("Vocabulary path:", VOCAB_PATH)
print("Tokenizer size:", len(tokenizer))
print("CTC blank/padding id:", tokenizer.pad_token_id)
print("Unknown-token id:", tokenizer.unk_token_id)
print("Vocabulary:", tokenizer.get_vocab())


Vocabulary path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_2/vocab.json
Tokenizer size: 34
CTC blank/padding id: 33
Unknown-token id: 32
Vocabulary: {'|': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṭ': 31, '[UNK]': 32, '[PAD]': 33}


In [ ]:
# Cell 7 — Verify that every V1.2 transcript is representable

unknown_segments = []

for row in frozen_df.itertuples(index=False):
    label_ids = tokenizer(row.transcription).input_ids
    if tokenizer.unk_token_id in label_ids:
        unknown_segments.append(
            (row.segment_id, row.transcription)
        )

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    for item in unknown_segments[:20]:
        print(item)

assert not unknown_segments, (
    "Some V1.2 references cannot be represented by the tokenizer."
)

print("All V1.2 references are representable.")


Segments containing [UNK]: 0
All V1.2 references are representable.


In [ ]:
# Cell 8 — Build or reload the cached V1.2 XLS-R dataset

import json
import soundfile as sf
from datasets import Dataset, DatasetDict, load_from_disk


def prepare_split(frame):
    work = frame.copy()
    work["absolute_audio_path"] = work["audio_path"].apply(
        lambda path: str(PROJECT_ROOT / path)
    )

    return Dataset.from_pandas(
        work[
            [
                "segment_id",
                "recording_id",
                "speaker_group_id",
                "duration_seconds",
                "absolute_audio_path",
                "transcription",
            ]
        ],
        preserve_index=False,
    )


def prepare_example(example):
    audio, sampling_rate = sf.read(
        example["absolute_audio_path"],
        dtype="float32",
        always_2d=False,
    )

    if sampling_rate != 16000:
        raise ValueError(
            f'{example["segment_id"]}: expected 16000 Hz, '
            f"found {sampling_rate} Hz"
        )

    if getattr(audio, "ndim", 1) != 1:
        raise ValueError(
            f'{example["segment_id"]}: audio is not mono'
        )

    model_inputs = processor(
        audio,
        sampling_rate=16000,
    )

    labels = tokenizer(
        example["transcription"]
    ).input_ids

    return {
        "input_values": model_inputs.input_values[0],
        "input_length": len(model_inputs.input_values[0]),
        "labels": labels,
    }


CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

cache_is_valid = False

if DATASET_CACHE_DIR.exists() and CACHE_MANIFEST_PATH.exists():
    with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as file:
        cache_manifest = json.load(file)

    cache_is_valid = (
        cache_manifest.get("metadata_sha256") == metadata_sha256
        and cache_manifest.get("tokenizer_size") == len(tokenizer)
        and cache_manifest.get("base_model") == BASE_MODEL_ID
    )

if DATASET_CACHE_DIR.exists() and not cache_is_valid:
    raise RuntimeError(
        "An XLS-R V1.2 cache exists but does not match the frozen metadata, "
        "tokenizer, or base model. Inspect it before choosing a new cache path."
    )

if cache_is_valid:
    xlsr_dataset = load_from_disk(str(DATASET_CACHE_DIR))
    print("Reused validated V1.2 XLS-R dataset cache.")
else:
    raw_dataset = DatasetDict(
        {
            "train": prepare_split(train_df),
            "validation": prepare_split(validation_df),
        }
    )

    xlsr_dataset = raw_dataset.map(
        prepare_example,
        remove_columns=[
            "absolute_audio_path",
            "transcription",
            "recording_id",
            "speaker_group_id",
            "duration_seconds",
        ],
        num_proc=1,
        desc="Preparing V1.2 16 kHz audio and CTC labels",
    )

    DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    xlsr_dataset.save_to_disk(str(DATASET_CACHE_DIR))

    with open(CACHE_MANIFEST_PATH, "w", encoding="utf-8") as file:
        json.dump(
            {
                "metadata_sha256": metadata_sha256,
                "tokenizer_size": len(tokenizer),
                "base_model": BASE_MODEL_ID,
            },
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("Built and saved V1.2 XLS-R dataset cache.")

train_ds = xlsr_dataset["train"]
validation_ds = xlsr_dataset["validation"]

print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Dataset columns:", train_ds.column_names)


Reused validated V1.2 XLS-R dataset cache.
Train examples: 1754
Validation examples: 129
Dataset columns: ['segment_id', 'input_values', 'input_length', 'labels']


In [ ]:
# Cell 9 — Summarize the processed V1.2 audio durations

def duration_summary(dataset_split):
    seconds = np.asarray(
        dataset_split["input_length"],
        dtype=np.float64,
    ) / 16000.0

    return {
        "segments": len(seconds),
        "hours": float(seconds.sum() / 3600),
        "minimum_seconds": float(seconds.min()),
        "maximum_seconds": float(seconds.max()),
        "mean_seconds": float(seconds.mean()),
    }


print("Train:", duration_summary(train_ds))
print("Validation:", duration_summary(validation_ds))


Train: {'segments': 1754, 'hours': 5.222733593749999, 'minimum_seconds': 0.848, 'maximum_seconds': 19.984, 'mean_seconds': 10.71940760404789}
Validation: {'segments': 129, 'hours': 0.2983577777777778, 'minimum_seconds': 0.944, 'maximum_seconds': 26.0, 'mean_seconds': 8.326263565891473}


In [ ]:
# Cell 10 — Load a fresh XLS-R 300M CTC model for V1.2

from transformers import Wav2Vec2ForCTC

# This diagnostic run intentionally uses no SpecAugment.
APPLY_SPEC_AUGMENT = False
MASK_TIME_PROB = 0.0
MASK_TIME_LENGTH = 5

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    apply_spec_augment=APPLY_SPEC_AUGMENT,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=0.0,
    layerdrop=0.0,
    ignore_mismatched_sizes=True,
)

# Keep the convolutional feature encoder frozen, as in the previous X2 run.
model.freeze_feature_encoder()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Base model:", BASE_MODEL_ID)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.4f}%",
)
print("External waveform augmentation: none")
print("Internal SpecAugment enabled:", model.config.apply_spec_augment)
print("Internal time-mask probability:", model.config.mask_time_prob)
print("Internal time-mask length:", model.config.mask_time_length)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model: facebook/wav2vec2-xls-r-300m
Total parameters: 315,472,546
Trainable parameters: 311,262,370
Trainable percentage: 98.6654%
External waveform augmentation: none
Internal SpecAugment enabled: False
Internal time-mask probability: 0.0
Internal time-mask length: 5


In [ ]:
# Cell 11 — Check CTC feasibility of every V1.2 example

def minimum_ctc_frames(labels):
    repeated_adjacent_labels = sum(
        labels[index] == labels[index - 1]
        for index in range(1, len(labels))
    )
    return len(labels) + repeated_adjacent_labels


def find_ctc_infeasible(dataset_split):
    invalid = []

    for index, example in enumerate(dataset_split):
        input_samples = int(example["input_length"])
        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(input_samples)
            ).item()
        )

        labels = example["labels"]
        required_frames = minimum_ctc_frames(labels)

        if output_frames < required_frames:
            invalid.append(
                {
                    "index": index,
                    "segment_id": example.get("segment_id", index),
                    "audio_seconds": input_samples / 16000.0,
                    "output_frames": output_frames,
                    "label_length": len(labels),
                    "minimum_ctc_frames": required_frames,
                }
            )

    return invalid


bad_train = find_ctc_infeasible(train_ds)
bad_validation = find_ctc_infeasible(validation_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_validation))

if bad_train:
    display(pd.DataFrame(bad_train))
if bad_validation:
    display(pd.DataFrame(bad_validation))

assert not bad_train, (
    "Training contains CTC-infeasible examples. Inspect before training."
)
assert not bad_validation, (
    "Validation contains CTC-infeasible examples. Inspect before training."
)

print("All V1.2 examples are CTC-feasible.")


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

CTC-infeasible training examples: 0
CTC-infeasible validation examples: 0
All V1.2 examples are CTC-feasible.


In [ ]:
# Cell 12 — Define dynamic CTC padding for audio and labels

from dataclasses import dataclass
from typing import Dict, List, Union


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("CTC data collator ready.")


CTC data collator ready.


In [ ]:
# Cell 13 — Define validation WER and CER metrics

from jiwer import wer, cer


def compute_metrics(prediction):
    prediction_ids = np.argmax(
        prediction.predictions,
        axis=-1,
    )

    label_ids = prediction.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    prediction_text = tokenizer.batch_decode(prediction_ids)
    reference_text = tokenizer.batch_decode(
        label_ids,
        group_tokens=False,
    )

    return {
        "wer": wer(reference_text, prediction_text) * 100,
        "cer": cer(reference_text, prediction_text) * 100,
    }


print("WER/CER metrics ready.")


WER/CER metrics ready.


In [ ]:
# Cell 14 — Save the XLS-R V1.2 experiment configuration

experiment_configuration = {
    "experiment": EXPERIMENT_NAME,
    "base_model": BASE_MODEL_ID,
    "augmentation_condition": AUGMENTATION_CONDITION,
    "metadata_path": str(FROZEN_METADATA_PATH),
    "metadata_sha256": metadata_sha256,
    "tokenizer_path": str(TOKENIZER_DIR),
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "seed": SEED,
    "external_waveform_augmentation": False,
    "internal_spec_augment": APPLY_SPEC_AUGMENT,
    "internal_mask_time_prob": MASK_TIME_PROB,
    "internal_mask_time_length": MASK_TIME_LENGTH,
    "feature_encoder_frozen": True,
    "planned_epochs": 8,
    "early_stopping": False,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "fp16": bool(torch.cuda.is_available()),
    "gradient_checkpointing": True,
    "checkpoint_selection_metric": "validation CER",
}

CONFIGURATION_PATH = RESULTS_DIR / "experiment_configuration.json"

with open(CONFIGURATION_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_configuration,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved configuration:", CONFIGURATION_PATH)


Saved configuration: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug_full8/experiment_configuration.json


In [ ]:
# Cell 15 — Configure the XLS-R V1.2 Trainer

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    group_by_length=True,
    length_column_name="input_length",
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    num_train_epochs=8,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

print(training_args)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [ ]:
# Cell 16 — Create the XLS-R V1.2 Trainer without early stopping

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=validation_ds,
    processing_class=processor,
    # Early stopping is intentionally disabled so all eight epochs run.
)

print("Trainer ready.")
print("Early stopping: disabled")
print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Planned epochs:", training_args.num_train_epochs)


Trainer ready.
Early stopping: disabled
Train examples: 1754
Validation examples: 129
Planned epochs: 8


In [ ]:
# Cell 17 — Start XLS-R V1.2 fine-tuning

train_result = trainer.train()

print("Training completed.")
print(train_result)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,3.916900,3.873901,100.000000,100.000000
2,3.060500,3.309227,100.000000,100.000000
3,2.943300,3.257802,100.000000,100.000000
4,2.906300,3.254106,100.000000,100.000000
5,2.635500,3.198607,99.957483,93.287242
6,2.280900,3.230296,100.340136,73.124849
7,1.981300,3.200279,101.105442,68.212879
8,1.844000,3.183398,101.785714,65.833266


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: N

Training completed.
TrainOutput(global_step=880, training_loss=3.23827085386623, metrics={'train_runtime': 1435.5079, 'train_samples_per_second': 9.775, 'train_steps_per_second': 0.613, 'total_flos': 4.566639967258468e+18, 'train_loss': 3.23827085386623, 'epoch': 8.0})
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug_full8/checkpoint-880
Best validation CER: 65.83326633973792


In [ ]:
# Cell 18 — Evaluate and save the best XLS-R V1.2 checkpoint

best_validation_metrics = trainer.evaluate(
    eval_dataset=validation_ds,
    metric_key_prefix="validation",
)

print("Best-checkpoint validation metrics:")
for key, value in best_validation_metrics.items():
    print(f"{key}: {value}")

BEST_MODEL_DIR = MODEL_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
processor.save_pretrained(BEST_MODEL_DIR)

print("Saved best model:", BEST_MODEL_DIR)


Best-checkpoint validation metrics:
validation_loss: 3.1833808422088623
validation_wer: 101.78571428571428
validation_cer: 65.81718787683897
validation_runtime: 4.0019
validation_samples_per_second: 32.234
validation_steps_per_second: 16.242
epoch: 8.0
Saved best model: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug_full8/best_model


In [ ]:
# Cell 19 — Save V1.2 validation predictions for error analysis

prediction_output = trainer.predict(
    validation_ds,
    metric_key_prefix="validation_prediction",
)

prediction_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

prediction_texts = tokenizer.batch_decode(prediction_ids)
reference_texts = [
    tokenizer.decode(
        example["labels"],
        group_tokens=False,
    )
    for example in validation_ds
]

validation_predictions = pd.DataFrame(
    {
        "segment_id": validation_ds["segment_id"],
        "reference": reference_texts,
        "prediction": prediction_texts,
    }
)

PREDICTIONS_PATH = RESULTS_DIR / "validation_predictions.csv"
validation_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(validation_predictions.head(20))
print("Saved predictions:", PREDICTIONS_PATH)


,segment_id,reference,prediction
0,REC090_SEG0010,ssalamuɛlikum necc meryem,raiennaaaitesama asuaaaenanni isemima tɣaa ann...
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,ien wumarɣariaaaamimaaamnni iraaraw aa aaiumin...
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,emuiarrwa n tawa inuarinaninirin aninaam uiat...
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,manacamaiu iden ar imeti miussa tesati n uratn...
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,uaamnn reniu iraaamamanaayisaenaia su iawa in...
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,mamiasureatawin raimmenntim udenwwann awaedens...
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,inamemu saraenm yanimem aaraid iwirara isennau...
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,sa ia manainnin waiinnnswacin arasananasaareim...
8,REC090_SEG0018,lmuhim wsiɣd,tar n aimewau nairiawau a ttamtuaam araasen ti...
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,amrmaasmiaɣasmimaiirarad sema maaiina a ua a a...


Saved predictions: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug_full8/validation_predictions.csv


In [ ]:
# Cell 20 — Save the training history and final experiment summary

history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = RESULTS_DIR / "training_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

summary = {
    **experiment_configuration,
    "early_stopping": False,
    "train_segments": len(train_ds),
    "validation_segments": len(validation_ds),
    "train_duration": duration_summary(train_ds),
    "validation_duration": duration_summary(validation_ds),
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_metric": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "final_validation_metrics": {
        key: float(value)
        if isinstance(value, (int, float, np.floating))
        else value
        for key, value in best_validation_metrics.items()
    },
    "results": {
        "training_history": str(HISTORY_PATH),
        "validation_predictions": str(PREDICTIONS_PATH),
        "best_model": str(BEST_MODEL_DIR),
    },
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"
with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print("Saved training history:", HISTORY_PATH)
print("Saved experiment summary:", SUMMARY_PATH)


Saved training history: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug_full8/training_history.csv
Saved experiment summary: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/xlsr_300m_tarifit_v1_2_noaug_full8/experiment_summary.json


In [ ]:
# Cell 21 — Diagnose XLS-R validation predictions

from jiwer import wer, cer

references = validation_predictions["reference"].tolist()
hypotheses = validation_predictions["prediction"].tolist()

space_free_references = [text.replace(" ", "") for text in references]
space_free_hypotheses = [text.replace(" ", "") for text in hypotheses]

empty_count = sum(not text.strip() for text in hypotheses)

print("Empty hypotheses:", empty_count, "/", len(hypotheses))
print("Empty-hypothesis rate:", round(100 * empty_count / len(hypotheses), 2), "%")
print("WER:", round(wer(references, hypotheses) * 100, 2), "%")
print("CER:", round(cer(references, hypotheses) * 100, 2), "%")
print(
    "CER without spaces:",
    round(cer(space_free_references, space_free_hypotheses) * 100, 2),
    "%",
)

display(validation_predictions.head(20))

Empty hypotheses: 0 / 129
Empty-hypothesis rate: 0.0 %
WER: 104.42 %
CER: 88.87 %
CER without spaces: 89.77 %


,segment_id,reference,prediction
0,REC090_SEG0010,ssalamuɛlikum necc meryem,raiennaaaitesama asuaaaenanni isemima tɣaa ann...
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,ien wumarɣariaaaamimaaamnni iraaraw aa aaiumin...
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,emuiarrwa n tawa inuarinaninirin aninaam uiat...
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,manacamaiu iden ar imeti miussa tesati n uratn...
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,uaamnn reniu iraaamamanaayisaenaia su iawa in...
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,mamiasureatawin raimmenntim udenwwann awaedens...
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,inamemu saraenm yanimem aaraid iwirara isennau...
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,sa ia manainnin waiinnnswacin arasananasaareim...
8,REC090_SEG0018,lmuhim wsiɣd,tar n aimewau nairiawau a ttamtuaam araasen ti...
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,amrmaasmiaɣasmimaiirarad sema maaiina a ua a a...


In [ ]:
# Cell 21 — Calculate XLS-R decoding diagnostics

from collections import Counter
from jiwer import wer as jiwer_wer, cer as jiwer_cer

diagnostic_df = validation_predictions.merge(
    validation_df[["segment_id", "recording_id", "speaker_group_id"]],
    on="segment_id",
    how="left",
    validate="one_to_one",
)

diagnostic_df["reference"] = diagnostic_df["reference"].fillna("").astype(str)
diagnostic_df["prediction"] = diagnostic_df["prediction"].fillna("").astype(str)

diagnostic_df["reference_no_spaces"] = diagnostic_df["reference"].str.replace(
    " ", "", regex=False
)
diagnostic_df["prediction_no_spaces"] = diagnostic_df["prediction"].str.replace(
    " ", "", regex=False
)

diagnostic_df["reference_chars"] = diagnostic_df["reference_no_spaces"].str.len()
diagnostic_df["prediction_chars"] = diagnostic_df["prediction_no_spaces"].str.len()
diagnostic_df["reference_words"] = diagnostic_df["reference"].str.split().str.len()
diagnostic_df["prediction_words"] = diagnostic_df["prediction"].str.split().str.len()

diagnostic_df["empty_prediction"] = (
    diagnostic_df["prediction"].str.strip().eq("")
)

diagnostic_df["char_length_ratio"] = np.where(
    diagnostic_df["reference_chars"] > 0,
    diagnostic_df["prediction_chars"] / diagnostic_df["reference_chars"],
    np.nan,
)

diagnostic_df["word_length_ratio"] = np.where(
    diagnostic_df["reference_words"] > 0,
    diagnostic_df["prediction_words"] / diagnostic_df["reference_words"],
    np.nan,
)


def longest_repeated_character_run(text):
    text = text.replace(" ", "")

    if not text:
        return 0

    longest = 1
    current = 1

    for previous, character in zip(text, text[1:]):
        if character == previous:
            current += 1
            longest = max(longest, current)
        else:
            current = 1

    return longest


diagnostic_df["longest_repeated_character_run"] = diagnostic_df[
    "prediction"
].map(longest_repeated_character_run)

references = diagnostic_df["reference"].tolist()
hypotheses = diagnostic_df["prediction"].tolist()

references_no_spaces = diagnostic_df["reference_no_spaces"].tolist()
hypotheses_no_spaces = diagnostic_df["prediction_no_spaces"].tolist()

empty_count = int(diagnostic_df["empty_prediction"].sum())

print("Validation segments:", len(diagnostic_df))
print("Empty hypotheses:", empty_count, "/", len(diagnostic_df))
print(
    "Empty-hypothesis rate:",
    round(100 * empty_count / len(diagnostic_df), 2),
    "%",
)
print("WER:", round(jiwer_wer(references, hypotheses) * 100, 2), "%")
print("CER:", round(jiwer_cer(references, hypotheses) * 100, 2), "%")
print(
    "CER without spaces:",
    round(jiwer_cer(references_no_spaces, hypotheses_no_spaces) * 100, 2),
    "%",
)
print(
    "Mean character-length ratio:",
    round(diagnostic_df["char_length_ratio"].mean(), 3),
)
print(
    "Median character-length ratio:",
    round(diagnostic_df["char_length_ratio"].median(), 3),
)
print(
    "Maximum repeated-character run:",
    int(diagnostic_df["longest_repeated_character_run"].max()),
)

prediction_characters = Counter(
    "".join(diagnostic_df["prediction_no_spaces"].tolist())
)

unexpected_prediction_characters = {
    character: count
    for character, count in prediction_characters.items()
    if character not in set(FINAL_LETTERS)
}

print(
    "Unexpected prediction characters:",
    unexpected_prediction_characters,
)

Validation segments: 129
Empty hypotheses: 0 / 129
Empty-hypothesis rate: 0.0 %
WER: 104.42 %
CER: 88.87 %
CER without spaces: 89.77 %
Mean character-length ratio: 1.004
Median character-length ratio: 0.563
Maximum repeated-character run: 5
Unexpected prediction characters: {}


In [ ]:
# Cell 22 — Inspect over-generation and repeated-character errors

review_columns = [
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "reference",
    "prediction",
    "char_length_ratio",
    "word_length_ratio",
    "longest_repeated_character_run",
]

pd.set_option("display.max_colwidth", 250)

print("Most over-generated predictions:")
display(
    diagnostic_df.sort_values(
        "char_length_ratio",
        ascending=False,
    )[review_columns].head(15)
)

print("Predictions with the longest repeated-character runs:")
display(
    diagnostic_df.sort_values(
        "longest_repeated_character_run",
        ascending=False,
    )[review_columns].head(15)
)

Most over-generated predictions:


,segment_id,recording_id,speaker_group_id,reference,prediction,char_length_ratio,word_length_ratio,longest_repeated_character_run
0,REC090_SEG0010,REC090,SPK007,ssalamuɛlikum necc meryem,raiennaaaitesama asuaaaenanni isemima tɣaa anni amennii m tin imar iar i airiiiraiariesnna iaa nni arie a aimmiaamuemaraaiiniiaa iasiesini man unin aa yaramnidtiraan ianniiiwaaa a immra anin uraeeesr,7.521739,9.000000,4
8,REC090_SEG0018,REC090,SPK007,lmuhim wsiɣd,tar n aimewau nairiawau a ttamtuaam araasen tinm imamse a armai muiae minian ea ima,6.272727,7.500000,2
39,REC090_SEG0049,REC090,SPK007,min ḍay yuqɛen,amanaamanam auminan intam atatedrmaimi aaauama wamasa a tawasearmaan waywaiasa wa,6.000000,3.333333,3
57,REC090_SEG0214,REC090,SPK007,qlil,inmmaarinin mam iraii,4.750000,3.000000,2
101,REC138_SEG0015,REC138,SPK010,yiwey itent s ṭraṭa,a amanaaweam niretrure n daaasar aritawa inuɣaria wadiriarisiseama aamni mammaman,4.500000,2.500000,3
64,REC090_SEG0223,REC090,SPK007,ggit mammec xezzaɣ necc,iraasmtemin eswaasusrarmaa tt ai nrariimam sn aaiiia iaamaaseni maainnira am itaa dumn nnamu,4.000000,3.250000,3
34,REC090_SEG0044,REC090,SPK007,a mermitussid lweqt nni necc,ia aaainraasesaaiim uarinmrmisawraimeanaiawa awarsaaaaia nmatm muimira etnnirairain yrasaitiraini,3.750000,1.600000,4
80,REC090_SEG0240,REC090,SPK007,mani wa ḍay zarr ḥed,minaasua semmi iraɣariu raari dairariaudirarima awaniaidiaaruania,3.750000,1.200000,2
109,REC138_SEG0023,REC138,SPK010,mya ḍi mya ad xasent yazzer,ar isa yen yn uaiam mmumemumin snnan sadaauarwi sammismmenn esenatiayaen eaaaur,3.136364,1.833333,3
99,REC138_SEG0013,REC138,SPK010,qqarnas ma ɣars duru ḍi tneyart,asawan nsditadasmiwasm as aannmanw esa mamnwatsediwrammn a siwrmnaiimmnmuueiraira,2.846154,1.333333,2


Predictions with the longest repeated-character runs:


,segment_id,recording_id,speaker_group_id,reference,prediction,char_length_ratio,word_length_ratio,longest_repeated_character_run
4,REC090_SEG0014,REC090,SPK007,a necc mammec ira djjix ḍi lmeɣrib wadji manayenni uffix dda,uaamnn reniu iraaamamanaayisaenaia su iawa in rar ira arinnnnniia iraasu disnnannmm,1.460000,1.000000,5
126,REC138_SEG0044,REC138,SPK010,lmuhimm iruḥ ɣars ɣar babas nni ittaras ḍifllah issidef it ila axirih innas aqa necc aqa igga axmi ḍ anewji waha innas qa xseɣ tamɣart nni ittrassen ittettsen ḍi tɣaɣart xseɣ ad kidneɣ t ad kidneɣ teqqim rami teqqim ṭemɣarṭ nni ikkes tcacciyt amm...,ieinaa aaaiia,0.049793,0.037736,5
38,REC090_SEG0048,REC090,SPK007,safi qqimeɣ tfekkaɣ qqaɣas manaya was zemmaɣ ca safi ruxa uḥreɣ ḍi manaya ḍ manayenni uca ira ɣaneɣ lmacakil attas umi aqqaneɣ ɣaneɣ lmacakil attas uca man macakil min dam yuqɛan,itrsimawaaiirsiaaaa amammiarɣarma i muriraariwarie ia iua ama iitainu a inamin aseddamiiate,0.547297,0.354839,5
43,REC090_SEG0053,REC090,SPK007,ɣesmu ira ɣari ɛaweḍ lmacakil ak ides thimɣarin nni ira taddfen teffɣen la mayemmi tassend lla mayemmi mi tggen muhim ira ɣari attas n macakil lex taxeyyaṭ nni maca amenni,uumisie nsusidiritta dannnnnmas raasaain ra teaisnirammisn aman taaisn,0.443662,0.266667,5
78,REC090_SEG0238,REC090,SPK007,ḍ xemmi tuɣ t truḍ niɣ tmenɣan akidem man tuɣ tegged iɛni s lebzawz nnem,maamsa nitan innauaiani aarya iam euemummnnasin asaaaaa mimaimuasyirurin iidemi,1.224138,0.600000,5
0,REC090_SEG0010,REC090,SPK007,ssalamuɛlikum necc meryem,raiennaaaitesama asuaaaenanni isemima tɣaa anni amennii m tin imar iar i airiiiraiariesnna iaa nni arie a aimmiaamuemaraaiiniiaa iasiesini man unin aa yaramnidtiraan ianniiiwaaa a immra anin uraeeesr,7.521739,9.000000,4
37,REC090_SEG0047,REC090,SPK007,xmi tiriɣ tɣimiɣ weḥdi uca tfekkaɣ qqaɣas rux qa ɛannec sbax maci ad ṭawa inu ad mɣan add azzun ca n leḥwayej i tiɛicen iwḍan da war zemman ca adggen manayenni,minirat eirwatewaa nar nn demaaa aiiinnaiaan imwa traimasreanuaimmas aa wdirammmnur,0.573643,0.322581,4
34,REC090_SEG0044,REC090,SPK007,a mermitussid lweqt nni necc,ia aaainraasesaaiim uarinmrmisawraimeanaiawa awarsaaaaia nmatm muimira etnnirairain yrasaitiraini,3.750000,1.600000,4
1,REC090_SEG0011,REC090,SPK007,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,ien wumarɣariaaaamimaaamnni iraaraw aa aaiuminminminnminsii raatarna,1.852941,0.857143,4
61,REC090_SEG0220,REC090,SPK007,yemma ira aqat ḍi lmeɣrib wa uzzemaɣ necc ad ḥkkiɣ lmacakil inu i yemma min ɣa ḍay teg yemma qa rux mammec tini baɛɛed ssenni lla lla necc mammec twariɣ,iirtmariamaaaa,0.113821,0.033333,4


In [ ]:
# Cell 23 — Compare errors by recording and speaker

recording_rows = []

for (recording_id, speaker_group_id), group in diagnostic_df.groupby(
    ["recording_id", "speaker_group_id"],
    dropna=False,
):
    group_references = group["reference"].tolist()
    group_hypotheses = group["prediction"].tolist()

    group_references_no_spaces = group["reference_no_spaces"].tolist()
    group_hypotheses_no_spaces = group["prediction_no_spaces"].tolist()

    recording_rows.append(
        {
            "recording_id": recording_id,
            "speaker_group_id": speaker_group_id,
            "segments": len(group),
            "empty_rate_percent": round(
                100 * group["empty_prediction"].mean(),
                2,
            ),
            "WER_percent": round(
                100 * jiwer_wer(
                    group_references,
                    group_hypotheses,
                ),
                2,
            ),
            "CER_percent": round(
                100 * jiwer_cer(
                    group_references,
                    group_hypotheses,
                ),
                2,
            ),
            "CER_without_spaces_percent": round(
                100 * jiwer_cer(
                    group_references_no_spaces,
                    group_hypotheses_no_spaces,
                ),
                2,
            ),
            "mean_char_length_ratio": round(
                group["char_length_ratio"].mean(),
                3,
            ),
        }
    )

recording_summary = pd.DataFrame(recording_rows).sort_values(
    "CER_without_spaces_percent"
)

display(recording_summary)

,recording_id,speaker_group_id,segments,empty_rate_percent,WER_percent,CER_percent,CER_without_spaces_percent,mean_char_length_ratio
0,REC090,SPK007,87,0.0,104.59,88.33,89.19,1.040
1,REC138,SPK010,42,0.0,103.89,90.58,91.59,0.928


In [ ]:
# Cell 24 — Reconcile trainer and diagnostic metrics

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

print("\nMetrics returned by trainer.predict:")
print({
    key: value
    for key, value in prediction_output.metrics.items()
    if "wer" in key.lower() or "cer" in key.lower()
})

print("\nMetrics recomputed from prediction_output:")
print(compute_metrics(prediction_output))

Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_noaug_full8/checkpoint-880
Best metric: 65.83326633973792

Metrics returned by trainer.predict:
{'validation_prediction_wer': 101.78571428571428, 'validation_prediction_cer': 65.84934480263686}

Metrics recomputed from prediction_output:
{'wer': 101.78571428571428, 'cer': 65.84934480263686}
